# MUSEmotion Colab Workflow

Run the GPU training pipeline: BERT fine-tuning on GoEmotions, official EMOPIA MIDI download and preprocessing, conditional Transformer training, MIDI generation, and optional Gradio inference. The default cells use smoke configs so the complete path can run on a Colab T4; swap in `configs/classifier.yaml`, `configs/music.yaml`, and `configs/inference.yaml` for longer full training runs.

In [ ]:
!git clone https://github.com/Alex-Xi-Chen/ECE1508-Deep-Generative-Models.git || true
%cd /content/ECE1508-Deep-Generative-Models
!git pull --ff-only
!pip install -r requirements.txt
!pip install -e . --no-deps

In [ ]:
import subprocess, torch

print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
print('cuda_device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO CUDA')
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True))

In [ ]:
!python -m pytest -q

## Download EMOPIA

This pulls the official `EMOPIA_1.0.zip` archive from Zenodo and extracts it to `data/raw/emopia/`.

In [ ]:
!python -m musemotion.cli.download_emopia --output data/raw/emopia

In [ ]:
!python -m musemotion.cli.train_classifier --config configs/colab_smoke_classifier.yaml

In [ ]:
!python -m musemotion.cli.prepare_emopia --config configs/colab_smoke_music.yaml

In [ ]:
!python -m musemotion.cli.train_generator --config configs/colab_smoke_music.yaml

In [ ]:
!python -m musemotion.cli.generate \
  --config configs/inference_colab_smoke.yaml \
  --text "I feel hopeful but calm today" \
  --output artifacts/colab_smoke/samples/hopeful_calm.mid \
  --seed 1508

In [ ]:
!python -m musemotion.frontend.app --config configs/inference_colab_smoke.yaml --share